# Titanic - Classification avec Machine Learning classique

# Imports

In [1]:
# data processing
import numpy as np
import pandas as pd

# machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# utils
import time
from datetime import timedelta

In [2]:
# Input files
file_train='../data/train.csv'
file_test='../data/test.csv'

In [3]:
seed = 69
np.random.seed(seed)

# Chargement des données

In [4]:
train_df = pd.read_csv(file_train, index_col='PassengerId')

In [5]:
# Show the columns
train_df.columns.values

array(['Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked'], dtype=object)

In [6]:
# Show the shape
train_df.shape

(891, 11)

In [7]:
# Preview the training data
train_df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [8]:
# Show NaN values
train_df.isnull().sum()

Survived      0
Pclass        0
Name          0
Sex           0
Age         177
SibSp         0
Parch         0
Ticket        0
Fare          0
Cabin       687
Embarked      2
dtype: int64

# Préparation des données
- Drop unwanted features ['Name', 'Ticket', 'Cabin']
- Fill missing data: Age and Fare with the mean, Embarked with most frequent value
- Convert categorical features into numeric
- Convert Embarked to one-hot

In [9]:
def prep_data(df):
    # Drop unwanted features
    df = df.drop(['Name', 'Ticket', 'Cabin'], axis=1)
    
    # Fill missing data: Age and Fare with the mean, Embarked with most frequent value
    df[['Age']] = df[['Age']].fillna(value=df[['Age']].mean())
    df[['Fare']] = df[['Fare']].fillna(value=df[['Fare']].mean())
    df[['Embarked']] = df[['Embarked']].fillna(value=df['Embarked'].value_counts().idxmax())
    
    # Convert categorical features into numeric
    df['Sex'] = df['Sex'].map({'female': 1, 'male': 0}).astype(int)
      
    # Convert Embarked to one-hot
    embarked_one_hot = pd.get_dummies(df['Embarked'], prefix='Embarked')
    df = df.drop('Embarked', axis=1)
    df = df.join(embarked_one_hot)

    return df

In [10]:
train_df = prep_data(train_df)
train_df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked_C    0
Embarked_Q    0
Embarked_S    0
dtype: int64

In [11]:
# X contains all columns except 'Survived'
X = train_df.drop(['Survived'], axis=1).values.astype(float)

# Scaling
scale = StandardScaler()
X = scale.fit_transform(X)

# Y is just the 'Survived' column
Y = train_df['Survived'].values

# Comparaison de plusieurs modèles ML

In [12]:
# Définition des modèles à comparer
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=seed),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=seed),
    'SVM': SVC(kernel='rbf', random_state=seed),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=seed),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=seed)
}

# Cross-validation pour chaque modèle
kfold = KFold(n_splits=10, shuffle=True, random_state=seed)

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, Y, cv=kfold, scoring='accuracy')
    results[name] = scores
    print(f'{name}: Mean Accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})')

Logistic Regression: Mean Accuracy = 0.7991 (+/- 0.0337)
Random Forest: Mean Accuracy = 0.8104 (+/- 0.0292)
SVM: Mean Accuracy = 0.8249 (+/- 0.0227)
KNN: Mean Accuracy = 0.8047 (+/- 0.0175)
Decision Tree: Mean Accuracy = 0.7756 (+/- 0.0350)
Gradient Boosting: Mean Accuracy = 0.8261 (+/- 0.0293)


# GridSearch sur le meilleur modèle (Random Forest)

In [13]:
run_gridsearch = False

if run_gridsearch:
    start_time = time.time()
    print(time.strftime("%H:%M:%S") + " GridSearch started...")
    
    param_grid = {
        'n_estimators': [50, 100, 200, 300],
        'max_depth': [3, 5, 7, 10, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    
    grid = GridSearchCV(
        estimator=RandomForestClassifier(random_state=seed),
        param_grid=param_grid,
        cv=kfold,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    grid_result = grid.fit(X, Y)
    
    print(f"\nBest: {grid_result.best_score_:.4f} using {grid_result.best_params_}")
    
    elapsed_time = time.time() - start_time
    print(f"Time elapsed: {timedelta(seconds=elapsed_time)}")
    
    best_model = grid_result.best_estimator_
    
else:
    # Paramètres pré-sélectionnés
    best_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=7,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=seed
    )

# Entraînement du modèle final et prédiction

In [14]:
# Entraîner le modèle final sur toutes les données d'entraînement
best_model.fit(X, Y)

# Cross-validation du modèle final
final_scores = cross_val_score(best_model, X, Y, cv=kfold, scoring='accuracy')
print(f'Modèle final - Mean Accuracy: {final_scores.mean():.4f} (+/- {final_scores.std():.4f})')

Modèle final - Mean Accuracy: 0.8238 (+/- 0.0273)


In [15]:
# Read test data
test_df = pd.read_csv(file_test, index_col='PassengerId')

# Prep test data
test_df = prep_data(test_df)

# Create X_test
X_test = test_df.values.astype(float)
# Scaling
X_test = scale.transform(X_test)

# Predict 'Survived'
prediction = best_model.predict(X_test)
print(f'Predictions shape: {prediction.shape}')
print(f'Survived: {prediction.sum()} / {len(prediction)}')

Predictions shape: (418,)
Survived: 135 / 418
